# Financial Fraud Detection & Transaction Risk Analysis

## Import Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [10]:
df = pd.read_csv("fraud_detection_dataset_200k.csv")

## Initial Data Exploration

In [3]:
df.head()

,transaction_id,customer_id,merchant_id,transaction_type,payment_method,device_type,city,amount,old_balance,new_balance,transaction_hour,transaction_date,customer_age,is_international,failed_attempts,account_tenure_months,is_fraud
0,TXN0000001,CUST93810,MER2824,TRANSFER,Wallet,Android,Lucknow,1642.44,96738.51,111646.39,7,2025-05-21 07:14:08,45,0,1,4,1
1,TXN0000002,CUST22280,MER4582,CASH_IN,Net Banking,iPhone,Ahmedabad,593.59,6872.91,6279.32,0,2025-11-05 00:35:12,55,1,3,104,0
2,TXN0000003,CUST10851,MER3615,PAYMENT,UPI,Android,Kolkata,12262.45,95590.49,83328.04,8,2025-06-24 08:09:13,24,1,0,109,0
3,TXN0000004,CUST55082,MER5333,DEBIT,UPI,Web,Lucknow,702.38,19451.13,18748.75,17,2025-08-24 17:07:59,41,0,0,91,0
4,TXN0000005,CUST19116,MER1750,TRANSFER,Net Banking,Web,Ahmedabad,2603.75,46082.28,43478.53,2,2025-05-29 02:54:14,58,1,0,21,0


In [4]:
df.shape

(201000, 17)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 201000 entries, 0 to 200999
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   transaction_id         201000 non-null  str    
 1   customer_id            201000 non-null  str    
 2   merchant_id            201000 non-null  str    
 3   transaction_type       201000 non-null  str    
 4   payment_method         199492 non-null  str    
 5   device_type            199489 non-null  str    
 6   city                   199497 non-null  str    
 7   amount                 201000 non-null  float64
 8   old_balance            201000 non-null  float64
 9   new_balance            201000 non-null  float64
 10  transaction_hour       201000 non-null  int64  
 11  transaction_date       201000 non-null  str    
 12  customer_age           201000 non-null  int64  
 13  is_international       201000 non-null  int64  
 14  failed_attempts        201000 non-null  int64  

In [6]:
df.describe()

,amount,old_balance,new_balance,transaction_hour,customer_age,is_international,failed_attempts,account_tenure_months,is_fraud
count,2.010000e+05,201000.000000,201000.000000,201000.000000,201000.000000,201000.000000,201000.000000,201000.000000,201000.000000
mean,3.742563e+03,53763.438383,50683.819938,11.514692,43.991726,0.499970,0.999562,60.337811,0.029965
std,9.702385e+03,28929.416154,28861.313908,6.918576,15.283767,0.500001,1.001886,34.635407,0.170492
min,1.000000e-02,511.690000,500.550000,0.000000,18.000000,0.000000,0.000000,1.000000,0.000000
25%,1.013638e+03,28878.460000,25789.777500,6.000000,31.000000,0.000000,0.000000,30.000000,0.000000
50%,2.428790e+03,53762.675000,50683.215000,12.000000,44.000000,0.000000,1.000000,60.000000,0.000000
75%,4.846795e+03,78595.925000,75526.152500,18.000000,57.000000,1.000000,2.000000,90.000000,0.000000
max,1.058871e+06,129594.580000,138236.170000,23.000000,70.000000,1.000000,8.000000,120.000000,1.000000


## Missing Value Analysis

In [7]:
df.isnull().sum()

transaction_id              0
customer_id                 0
merchant_id                 0
transaction_type            0
payment_method           1508
device_type              1511
city                     1503
amount                      0
old_balance                 0
new_balance                 0
transaction_hour            0
transaction_date            0
customer_age                0
is_international            0
failed_attempts             0
account_tenure_months       0
is_fraud                    0
dtype: int64

## Handle Missing Values

In [14]:
df['device_type']= df['device_type'].fillna(df['device_type'].mode()[0])

df['payment_method']=df['payment_method'].fillna(df['payment_method'].mode()[0])

df['city']=df['city'].fillna(df['city'].mode()[0])

## Check Duplicate Records

In [16]:
df.duplicated().sum()

np.int64(991)

## Removing Duplicate Records

In [17]:
df = df.drop_duplicates()

## Text Standardization

In [19]:
df['transaction_type'] = df['transaction_type'].str.strip()

## Datetime Conversion

In [20]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

## Feature Engineering

In [21]:
df['transaction_day'] = df['transaction_date'].dt.day_name()

df['transaction_month'] = df['transaction_date'].dt.month

df['transaction_year'] = df['transaction_date'].dt.year

In [22]:
df.to_csv("cleaned_fraud_dataset.csv", index=False)
print("successfully saved")

successfully saved


In [11]:
!pip install mysql-connector-python sqlalchemy pymysql


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sqlalchemy import create_engine

In [13]:
username = "root"
password = "Password"
host = "localhost"
database = "fraud_project"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

In [14]:
df.to_sql(
    name='fraud_transactions',
    con=engine,
    if_exists='replace',
    index=False
)

201000

In [15]:
query = "SELECT * FROM fraud_transactions LIMIT 5"

pd.read_sql(query, engine)

,transaction_id,customer_id,merchant_id,transaction_type,payment_method,device_type,city,amount,old_balance,new_balance,transaction_hour,transaction_date,customer_age,is_international,failed_attempts,account_tenure_months,is_fraud
0,TXN0000001,CUST93810,MER2824,TRANSFER,Wallet,Android,Lucknow,1642.44,96738.51,111646.39,7,2025-05-21 07:14:08,45,0,1,4,1
1,TXN0000002,CUST22280,MER4582,CASH_IN,Net Banking,iPhone,Ahmedabad,593.59,6872.91,6279.32,0,2025-11-05 00:35:12,55,1,3,104,0
2,TXN0000003,CUST10851,MER3615,PAYMENT,UPI,Android,Kolkata,12262.45,95590.49,83328.04,8,2025-06-24 08:09:13,24,1,0,109,0
3,TXN0000004,CUST55082,MER5333,DEBIT,UPI,Web,Lucknow,702.38,19451.13,18748.75,17,2025-08-24 17:07:59,41,0,0,91,0
4,TXN0000005,CUST19116,MER1750,TRANSFER,Net Banking,Web,Ahmedabad,2603.75,46082.28,43478.53,2,2025-05-29 02:54:14,58,1,0,21,0
